<a href="https://colab.research.google.com/github/AytenHarman07/AytenHarman07/blob/main/Clinical_Mutation_Hunter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install biopython

"""
Klinik Mutasyon taraması: Gerçek NCBI Verisi ile Nokta Mutasyonu Tespiti
Yazar: Ayten
Açıklama: Bu kod, NCBI'dan gerçek TP53 genini çeker, dijital ortamda
kanserli bir hasta verisi simüle eder ve iki devasa diziyi karşılaştırarak
nokta mutasyonunun (Point Mutation) tam adresini bulur.
"""

from Bio import Entrez, SeqIO

# Entrez'e kurum kimliğimizi gösteriyoruz
Entrez.email = "harmanayten07@gmail.com"

def gercek_veri_mutasyon_taramasi():
    accession_id = "NM_000546" # İnsan p53 Tümör Baskılayıcı Geni

    # --- 1. ADIM: GERÇEK SAĞLIKLI VERİYİ ÇEKME ---
    print(f"[*] NCBI sistemine bağlanılıyor... {accession_id} çekiliyor.")
    try:
        handle = Entrez.efetch(db="nucleotide", id=accession_id, rettype="fasta", retmode="text")
        saglikli_kayit = SeqIO.read(handle, "fasta")
        handle.close()

        # DNA objesini, harf harf analiz edebilmek için düz metne (string) çeviriyoruz
        saglikli_dizi = str(saglikli_kayit.seq)
        print(f"[+] Başarılı! {len(saglikli_dizi)} nükleotid uzunluğunda sağlıklı referans gen elde edildi.\n")

    except Exception as e:
        print(f"[-] Hata oluştu, veritabanına ulaşılamadı: {e}")
        return

    # --- 2. ADIM: KANSERLİ HASTA VERİSİ YARATMA (SİMÜLASYON) ---
    # Sağlıklı genin birebir kopyasını alıp, hastanın genini oluşturuyoruz.
    hasta_dizi_listesi = list(saglikli_dizi)

    # BİLEREK MUTASYON YAPIYORUZ: 814. indeksteki (gerçekte 815. nükleotid) "G" harfini "T" yapıyoruz.
    hasta_dizi_listesi[814] = "T"

    # Listeyi tekrar birleşik bir gen zinciri haline getiriyoruz
    hasta_dizi = "".join(hasta_dizi_listesi)

    # --- 3. ADIM: MUTASYON TARAMASI (DİJİTAL HİZALAMA) ---
    print("-" * 55)
    print("🔍 KLİNİK MUTASYON TARAMASI BAŞLATILDI")
    print("-" * 55)

    mutasyon_sayisi = 0

    # Önce kalite kontrol: İki genin boyu aynı mı?
    if len(saglikli_dizi) != len(hasta_dizi):
        print("Hata: Dizilerin uzunlukları farklı, çerçeve kayması (Frameshift) olabilir!")
        return

    # Dijital mikroskobumuz 2500 harfin üzerinden tek tek geçiyor
    for i in range(len(saglikli_dizi)):
        s_harf = saglikli_dizi[i]
        h_harf = hasta_dizi[i]

        # Eğer harfler eşleşmiyorsa, alarm ver!
        if s_harf != h_harf:
            print(f"🚨 [KIRMIZI KOD] {i+1}. Nükleotidde Nokta Mutasyonu Yakalandı!")
            print(f"   >>> Sağlıklı İnsan (Referans) : {s_harf}")
            print(f"   >>> Kanserli Hasta (Örnek)    : {h_harf}\n")
            mutasyon_sayisi += 1

    # Sonuç Raporu
    if mutasyon_sayisi == 0:
        print("✅ Hastanın geni referans dizi ile %100 eşleşiyor. Mutasyon yok.")
    else:
        print(f"⚠️ Toplam {mutasyon_sayisi} adet mutasyon raporlandı.")
        print("-" * 55)
        print("Analiz Tamamlandı.")

# Sistemin Düğmesine Basıyoruz
gercek_veri_mutasyon_taramasi()

[*] NCBI sistemine bağlanılıyor... NM_000546 çekiliyor.
[+] Başarılı! 2512 nükleotid uzunluğunda sağlıklı referans gen elde edildi.

-------------------------------------------------------
🔍 KLİNİK MUTASYON TARAMASI BAŞLATILDI
-------------------------------------------------------
🚨 [KIRMIZI KOD] 815. Nükleotidde Nokta Mutasyonu Yakalandı!
   >>> Sağlıklı İnsan (Referans) : G
   >>> Kanserli Hasta (Örnek)    : T

⚠️ Toplam 1 adet mutasyon raporlandı.
-------------------------------------------------------
Analiz Tamamlandı.
